# 验证小鼠文献 marker-high 细胞类型重命名候选

本 notebook 按 `test_0530.ipynb` 的验证方式写成小鼠版：用文献整理出的 `marker-high cell type -> marker genes` 字典，在整合后的 mouse AnnData 对象中检查 marker 表达、DotPlot 和 UMAP 分布。

主要用途：

1. 例如要 rename `T cell`，就在 `MOUSE_MARKER_SETS["T cell"]` 中放入各类 `marker-high T cell` 的 marker。
2. 在你的数据中筛选父类细胞，例如 `cell_type_level1_corrected == "T cell"`。
3. 检查 marker 是否存在于 `adata.var_names`。
4. 按当前注释列画 DotPlot，观察 marker 是否在某些现有 cluster 中富集。
5. 计算每个候选状态的 signature score，并在 UMAP 上查看空间分布。
6. 最后按需要重命名的 10 个父类细胞逐类验证：`T cell`、`Macrophage`、`Monocyte`、`Smooth muscle cell`、`Endothelial cell`、`Mast cell`、`Dendritic cell`、`B cell`、`Natural killer cell`、`Fibroblast`。

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", fontsize=8)

BASE_DIR = Path("/home/lixiangyu/zr/Annotate/ANNOTATE_new/10_downstream/cell_type_rename/test_0530")
FIG_DIR = BASE_DIR / "figures_mouse"
TABLE_DIR = BASE_DIR / "tables_mouse"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

MOUSE_EVIDENCE_TSV = BASE_DIR / "mouse_literature_celltypes" / "mouse_celltype_marker_evidence.tsv"

MOUSE_H5AD = Path("/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_mouse/scPoli_concat_level3_marker_allmouse.h5ad")

RANDOM_STATE = 7
MAX_CELLS_FOR_PLOT = 120_000
MIN_GENES = 2

BASE_DIR

## 读取小鼠文献证据表

这些表用于追溯每个候选命名来自哪篇文章。真正用于画图的是下面的 marker 字典；字典中的 marker 主要来自 `Definition_markers` 和 `Validation_markers`。

In [ ]:
evidence = pd.read_csv(MOUSE_EVIDENCE_TSV, sep="\t")
evidence[[
    "GSE",
    "Author_cell_type_or_state",
    "Parent_lineage",
    "Evidence_level",
    "Validation_method",
    "Source_url",
]].head(20)

In [ ]:
# 优先使用这些证据等级来做最终 rename 判断。
STRONG_EVIDENCE_LEVELS = [
    "全文支持",
    "全文支持但需验证",
]

strong_evidence = evidence[evidence["Evidence_level"].isin(STRONG_EVIDENCE_LEVELS)].copy()
strong_evidence[["GSE", "Author_cell_type_or_state", "Parent_lineage", "Evidence_level"]]

## 读取 AnnData

这里沿用 `test_0530.ipynb` 的方式：读取 mouse `.h5ad` 后，把 `var_names` 设置成 `original_gene_names`，这样 marker 名称可以直接按 mouse gene symbol 匹配。

In [ ]:
adata_mouse = sc.read_h5ad(MOUSE_H5AD)
adata_mouse.var["ensemble_id"] = adata_mouse.var_names
adata_mouse.var_names = adata_mouse.var["original_gene_names"].astype(str).values
adata_mouse.var_names_make_unique()
adata_mouse

In [ ]:
# 快速检查 metadata，确认细胞类型列和 UMAP 坐标是否存在。
for name, adata in {"mouse": adata_mouse}.items():
    print(f"\n## {name}")
    print("数据维度:", adata.shape)
    print("obsm 中的键:", list(adata.obsm.keys())[:10])
    print("obs 中可用列:", [c for c in ["dataset", "sample", "cell_type_level1_corrected", "cell_type_level1", "cell_type_level2", "cell_type_level3", "Plaque_type_pred", "Plaque_type_pred_3class", "symptoms"] if c in adata.obs.columns])
    for col in ["cell_type_level1_corrected", "cell_type_level1", "cell_type_level2", "cell_type_level3"]:
        if col in adata.obs.columns:
            print(f"\n{name} {col}:")
            print(adata.obs[col].astype(str).value_counts().head(30).to_string())

## marker-high 候选字典

字典结构是：

```python
{
    "父类细胞": {
        "候选 rename 名称": ["marker1", "marker2", ...]
    }
}
```

例如验证 T cell 时，使用 `MOUSE_MARKER_SETS["T cell"]`。

In [ ]:
MOUSE_MARKER_SETS = {
    "T cell": {
        # GSE245373, GSE253247, GSE239591
        "Cd3e-Il7r-high CD4 T cell": ["Cd3e", "Cd3d", "Cd4", "Il7r", "Ltb", "Ccr7"],
        "Cd8a-Nkg7-high CD8 T cell": ["Cd3e", "Cd3d", "Cd8a", "Nkg7", "Gzma", "Gzmb", "Prf1"],
        "Foxp3-Il2ra-high Treg": ["Foxp3", "Il2ra", "Ctla4", "Tnfrsf4", "Tnfrsf18", "Tigit"],
        "Mki67-high proliferating Treg/T cell": ["Mki67", "Top2a", "Stmn1", "Foxp3", "Il2ra"],
        "Cd69-high activated/memory T cell": ["Cd69", "Cd3e", "Cd3d", "Cd4", "Cd8a", "Il7r", "Nkg7"],
        "Spp1-Ctsb-Fth1-high senescent T cell": ["Spp1", "Ctsb", "Fth1", "Thbs1", "Tnfrsf11b", "Cst3", "Ctss"],
    },
    "Macrophage": {
        # GSE164678, GSE239620, GSE262694, GSE264071, GSE279601, GSE284253, GSE298574
        "Mrc1-Pf4-Cd163-high resident/phagocytic macrophage": ["Mrc1", "Pf4", "Adgre1", "Cd163", "C1qa", "C1qb", "C1qc"],
        "Il1b-H2-Ab1-high inflammatory macrophage": ["Il1b", "H2-Ab1", "H2-Aa", "Cebpb", "Srgn", "Aldoa", "Gpr35"],
        "Mki67-high proliferative macrophage": ["Mki67", "Top2a", "Stmn1", "Adgre1", "Cd68"],
        "Sting1-Ifnar1-Irf7-high IFNIC macrophage": ["Sting1", "Ifnar1", "Irf7", "Isg15", "Ifit1", "Ifit3", "Rsad2", "Cxcl10"],
        "Srsf3-Aars2-chemokine-high transitional macrophage": ["Srsf3", "Aars2", "Ccl2", "Ccl3", "Ccl4", "Cxcl2", "Cxcl10"],
        "Asxl1-Irak1-Nfkb-high inflammatory macrophage": ["Asxl1", "Irak1", "Map3k7", "Nfkb1", "Nfkbia", "Tnf", "Il1b"],
        "Trem2-Csf1-Spp1-high plaque macrophage": ["Trem2", "Csf1", "Csf1r", "Spp1", "Mmp9", "Ctsb", "Ctss", "Apoe", "Lpl"],
        "Kif13b-Mertk-high efferocytic macrophage": ["Kif13b", "Mertk", "Itch", "Cbl", "Axl", "Cd68", "Adgre1"],
        "Timd4-MHCII-high tissue-resident macrophage": ["Timd4", "H2-Ab1", "H2-Aa", "Adgre1", "Cx3cr1", "Cd68"],
    },
    "Monocyte": {
        # GSE164678, GSE237067, GSE262694, GSE264071
        "Ly6c2-Ccr2-high classical monocyte": ["Ly6c2", "Ccr2", "S100a8", "S100a9", "Lyz2", "Vcan"],
        "Cx3cr1-Nr4a1-high non-classical monocyte": ["Cx3cr1", "Nr4a1", "Fcgr4", "Itgal", "Lyz2"],
        "Ccl-Cxcl-high inflammatory transitional monocyte": ["Ccl2", "Ccl3", "Ccl4", "Cxcl2", "Cxcl10", "Srsf3"],
    },
    "Smooth muscle cell": {
        # SMC / VSMC: GSE131776, GSE141031, GSE155513, GSE191226, GSE205930, GSE209525, GSE210406, GSE233625, GSE239591, GSE264071, GSE264253, GSE269449, GSE274572
        "Lum-Fn1-high fibromyocyte/modulated SMC": ["Lum", "Fn1", "Dcn", "Bgn", "Col1a1", "Col1a2", "Lgals3", "Spp1", "Tcf21"],
        "Ly6a-Vcam1-Ly6c1-high SEM/intermediate SMC": ["Ly6a", "Vcam1", "Ly6c1", "Lgals3", "Fn1", "Klf4", "Tnfrsf11b"],
        "Myh11-Acta2-Cnn1-high contractile SMC": ["Myh11", "Acta2", "Cnn1", "Tagln", "Myl9", "Myocd"],
        "Eng-Nt5e-Thy1-high MSC-like SMC": ["Eng", "Nt5e", "Thy1", "Cd44", "Ly6a", "Myh11"],
        "Runx2-Spp1-Ibsp-high osteoblast-like SMC": ["Runx2", "Spp1", "Ibsp", "Alpl", "Bglap", "Col1a1"],
        "Sox9-Acan-Col2a1-high chondrocyte-like SMC": ["Sox9", "Acan", "Col2a1", "Comp", "Fmod"],
        "Pparg-Adipoq-high adipocyte-like SMC": ["Pparg", "Adipoq", "Fabp4", "Lpl"],
        "Cd68-Lgals3-high macrophage-like SMC": ["Cd68", "Lgals3", "Apoe", "Ctsb", "Ctsd", "Vcam1"],
        "Malat1-Mmp3-high VSMC4": ["Malat1", "Mmp3", "Mmp2", "Spp1", "Myh11", "Rock1"],
        "Dpp4-Cdkn1a-high senescent VSMC": ["Dpp4", "Cdkn1a", "Cdkn2a", "Trp53", "Gdf15", "Il1b", "Serpine1", "C3", "C2", "Acta2"],
        "Gsdme-Casp3-high pyroptotic/senescent VSMC": ["Gsdme", "Casp3", "Cdkn1a", "Cdkn2a", "Trp53", "Mmp9", "Ccl2", "Il1b", "Il18", "Acta2"],
        "Spp1-Serpine2-Ctsb-high senescent VSMC": ["Spp1", "Serpine2", "Ctsb", "Fth1", "Thbs1", "Tnfrsf11b", "Cst3", "Ctss"],
        "Icam1-Tnfrsf11b-Fmod-high premature senescent SMC": ["Icam1", "Tnfrsf11b", "Fmod", "Sfrp4", "Cdkn2a", "Cdkn1a", "Acta2", "Myh11"],
        "Notch3-F2r-high fibrous-cap VSMC": ["Notch3", "F2r", "Myh11", "Cnn1", "Acta2", "Col1a1", "Col1a2", "Eln", "Ccn2", "Thbs1", "Ankrd1"],
    },
    "Endothelial cell": {
        # GSE134557, GSE191226, GSE205930, GSE211216, GSE231306
        "Ccl2-Icam1-Vcam1-high inflammatory EC": ["Ccl2", "Icam1", "Vcam1", "Mmp2", "Fn1", "Tlr4", "Myd88"],
        "Fn1-Col1a1-Vim-high EndMT EC": ["Fn1", "Col1a1", "Col1a2", "Vim", "Tagln", "Acta2", "Pecam1", "Cdh5"],
        "Bmp4-Bgn-Irf6-Hoxa9-high injury EC": ["Bmp4", "Bgn", "Irf6", "Hoxa9", "Pecam1", "Cdh5", "Cldn5"],
        "Calcrl-high disease-associated EC": ["Calcrl", "Pecam1", "Cdh5", "Cldn5", "Kdr"],
        "Vcam1-Scarb2-high EC1": ["Vcam1", "Scarb2", "Pecam1", "Cdh5", "Cldn5"],
        "Fos-Jun-high AP1 EC program": ["Fos", "Jun", "Jund", "Atf3", "Icam1", "Vcam1", "Ccl2"],
    },
    "Mast cell": {
        # GSE245373
        "Kit-Hdc-Cma1-high mast cell": ["Kit", "Hdc", "Cma1", "Ms4a2", "Tpsb2", "Cpa3"],
    },
    "Dendritic cell": {
        # GSE237067, GSE245373
        "Xcr1-Clec9a-high cDC1": ["Xcr1", "Clec9a", "Batf3", "Irf8", "Flt3", "Cadm1"],
        "Itgax-Sirpa-Cd209a-high cDC2": ["Itgax", "Sirpa", "Cd209a", "Flt3", "H2-Ab1", "H2-Aa"],
        "Siglech-Bst2-Tcf4-high pDC": ["Siglech", "Bst2", "Tcf4", "Irf8", "Clec4c", "Flt3"],
    },
    "B cell": {
        # GSE245373
        "Cd79a-Ms4a1-high B cell": ["Cd79a", "Ms4a1", "Cd74", "H2-Ab1", "H2-Aa"],
        "Jchain-Mzb1-high plasma cell": ["Jchain", "Mzb1", "Xbp1", "Sdc1", "Ighm"],
    },
    "Natural killer cell": {
        # GSE245373
        "Nkg7-Klrd1-high NK cell": ["Nkg7", "Klrd1", "Gzma", "Gzmb", "Prf1", "Klrk1"],
        "Cd3e-Nkg7-high NK T cell": ["Cd3e", "Cd3d", "Cd8a", "Nkg7", "Klrd1"],
    },
    "Fibroblast": {
        # GSE164678, GSE191226, GSE205930, GSE239591
        "Col1a1-Col3a1-high ECM fibroblast": ["Col1a1", "Col1a2", "Col3a1", "Dcn", "Lum", "Fbln1"],
        "Mki67-high proliferative/inflammatory fibroblast": ["Mki67", "Top2a", "Stmn1", "Ccl2", "Il6", "Cxcl1"],
        "Spp1-Comp-Ltbp2-high senescent fibroblast": ["Spp1", "Comp", "Ltbp2", "Lum", "Ctsb", "Fth1", "Tnfrsf11b"],
        "Smoc2-high disease-associated fibroblast": ["Smoc2", "Pdgfra", "Fbln1", "Dcn", "Serpinf1"],
        "Fbn1-high segment-enriched fibroblast": ["Fbn1", "Dcn", "Lum", "Col1a1", "Col3a1"],
    },
}

list(MOUSE_MARKER_SETS.keys())

## 辅助函数

`run_validation(...)` 是核心函数：筛选父类细胞、检查 marker 覆盖率、画 DotPlot、计算 signature score，并保存结果表。

In [ ]:
def _normalize_values(values):
    if values is None:
        return None
    if isinstance(values, str):
        return [values]
    return list(values)


def show_evidence(parent_keyword=None, state_keyword=None, levels=None, table=None):
    """按父类细胞、候选状态和证据等级筛选小鼠文献证据表。"""
    df = evidence.copy() if table is None else table.copy()
    if df.empty:
        return df
    if parent_keyword and "Parent_lineage" in df.columns:
        m = df["Parent_lineage"].astype(str).str.contains(parent_keyword, case=False, regex=True, na=False)
        df = df[m]
    if state_keyword and "Author_cell_type_or_state" in df.columns:
        m = df["Author_cell_type_or_state"].astype(str).str.contains(state_keyword, case=False, regex=True, na=False)
        df = df[m]
    if levels and "Evidence_level" in df.columns:
        df = df[df["Evidence_level"].isin(_normalize_values(levels))]
    cols = [
        "GSE", "Article", "Author_cell_type_or_state", "Parent_lineage",
        "Definition_markers", "Validation_markers", "Evidence_level", "Evidence_note", "Source_url",
    ]
    cols = [c for c in cols if c in df.columns]
    return df[cols]


def subset_parent(adata, parent_values, parent_col="cell_type_level1_corrected", copy=True):
    """按父类细胞筛选数据，例如只保留 T cell 或 Macrophage。"""
    parent_values = _normalize_values(parent_values)
    if parent_values is None:
        return adata.copy() if copy else adata
    if parent_col not in adata.obs.columns:
        raise KeyError(f"{parent_col!r} 不在 adata.obs 中")
    mask = adata.obs[parent_col].astype(str).isin(parent_values)
    out = adata[mask]
    return out.copy() if copy else out


def filter_marker_dict(marker_dict, var_names, min_genes=1, verbose=True):
    """保留 AnnData 中存在的 marker，并报告缺失基因。"""
    var_names = pd.Index(var_names.astype(str) if hasattr(var_names, "astype") else var_names)
    out = {}
    report = []
    for state, genes in marker_dict.items():
        genes = list(dict.fromkeys(genes))
        present = [g for g in genes if g in var_names]
        missing = [g for g in genes if g not in var_names]
        report.append({
            "state": state,
            "n_total": len(genes),
            "n_present": len(present),
            "pct_present": len(present) / len(genes) if genes else 0,
            "present": ",".join(present),
            "missing": ",".join(missing),
        })
        if len(present) >= min_genes:
            out[state] = present
        if verbose:
            print(f"{state}: {len(present)}/{len(genes)} 个 marker 存在")
            if missing:
                print("  缺失:", ", ".join(missing))
    return out, pd.DataFrame(report)


def downsample_for_plot(adata, max_cells=None, random_state=None):
    """细胞数过多时下采样，避免 UMAP 和 DotPlot 过慢。"""
    if max_cells is None:
        max_cells = globals().get("MAX_CELLS_FOR_PLOT", 120_000)
    if random_state is None:
        random_state = globals().get("RANDOM_STATE", 7)
    if adata.n_obs <= max_cells:
        return adata
    return sc.pp.subsample(adata, n_obs=max_cells, random_state=random_state, copy=True)


def safe_state_name(state):
    """把候选状态名称转成适合用于列名和文件名的字符串。"""
    return (
        state.replace("/", "_")
        .replace(" ", "_")
        .replace("+", "pos")
        .replace("-", "_")
        .replace(",", "")
        .replace(";", "")
        .replace("(", "")
        .replace(")", "")
    )[:140]


def score_column_name(score_prefix, state):
    """根据前缀和候选状态生成 signature score 列名。"""
    return f"{score_prefix}_{safe_state_name(state)}"


def add_signature_scores(adata, marker_dict, score_prefix="score"):
    """用 scanpy score_genes 为每个候选状态计算 signature score。"""
    score_cols = {}
    for state, genes in marker_dict.items():
        col = score_column_name(score_prefix, state)
        sc.tl.score_genes(adata, gene_list=genes, score_name=col, use_raw=False)
        score_cols[state] = col
    return score_cols


def score_cols_list(score_cols):
    """兼容字典和列表两种 score 列输入。"""
    return list(score_cols.values()) if isinstance(score_cols, dict) else list(score_cols)


def score_summary_by_group(adata, score_cols, groupby):
    """按现有注释分组，汇总每个候选状态的平均 signature score。"""
    if groupby not in adata.obs.columns:
        raise KeyError(f"{groupby!r} 不在 adata.obs 中")
    return (
        adata.obs[[groupby] + score_cols]
        .groupby(groupby, observed=True)
        .mean()
        .sort_index()
    )


def split_marker_dict(marker_dict, max_genes=60):
    """DotPlot marker 太多时分块，避免图过宽。"""
    chunks = []
    current = {}
    n_genes = 0
    for state, genes in marker_dict.items():
        if current and n_genes + len(genes) > max_genes:
            chunks.append(current)
            current = {}
            n_genes = 0
        current[state] = genes
        n_genes += len(genes)
    if current:
        chunks.append(current)
    return chunks


def plot_dotplot(adata, marker_dict, groupby, title=None, figsize=None, save_path=None):
    """按当前注释分组绘制 marker DotPlot。"""
    if groupby not in adata.obs.columns:
        raise KeyError(f"{groupby!r} 不在 adata.obs 中")
    figures = []
    chunks = split_marker_dict(marker_dict, max_genes=60)
    for i, chunk in enumerate(chunks, start=1):
        n_genes = sum(len(v) for v in chunk.values())
        if figsize is None:
            chunk_figsize = (max(6, min(24, n_genes * 0.30)), max(4, 0.30 * adata.obs[groupby].nunique()))
        else:
            chunk_figsize = figsize
        if title:
            print(f"{title} - part {i}/{len(chunks)}")
        dp = sc.pl.dotplot(
            adata,
            groupby=groupby,
            var_names=chunk,
            standard_scale="var",
            swap_axes=False,
            figsize=chunk_figsize,
            return_fig=True,
        )
        dp.legend(width=2.2)
        if save_path:
            save_path = Path(save_path)
            suffix = f"_part{i}" if len(chunks) > 1 else ""
            dp.savefig(str(save_path.with_name(save_path.stem + suffix + save_path.suffix)))
        dp.show()
        figures.append(dp)
    return figures


def plot_umap_markers_and_scores(
    adata,
    marker_dict,
    state,
    score_col=None,
    max_marker_genes=8,
    color_map="viridis",
    save_prefix=None,
):
    """绘制一个候选状态的 marker 基因表达和 signature score 的 UMAP 分布。"""
    if "X_umap" not in adata.obsm:
        raise KeyError("adata.obsm 中没有 X_umap")
    if state not in marker_dict:
        raise KeyError(f"{state!r} 不在 marker 字典中。可选状态: {list(marker_dict)}")
    genes = marker_dict[state][:max_marker_genes]
    colors = genes.copy()
    if score_col:
        colors.append(score_col)
    axes = sc.pl.umap(
        adata,
        color=colors,
        cmap=color_map,
        ncols=3,
        frameon=False,
        wspace=0.3,
        save=False,
        show=False,
    )
    if save_prefix:
        plt.savefig(f"{save_prefix}.pdf", bbox_inches="tight")
        plt.savefig(f"{save_prefix}.png", bbox_inches="tight", dpi=200)
    plt.show()
    return axes


def plot_score_umap(adata, score_cols, parent, species="mouse", max_cols_per_fig=12):
    """绘制一个父类细胞所有候选状态的 signature score UMAP。"""
    cols = score_cols_list(score_cols)
    for i in range(0, len(cols), max_cols_per_fig):
        chunk = cols[i:i + max_cols_per_fig]
        axes = sc.pl.umap(
            adata,
            color=chunk,
            cmap="viridis",
            ncols=3,
            frameon=False,
            wspace=0.3,
            save=False,
            show=False,
        )
        suffix = f"_part{i // max_cols_per_fig + 1}" if len(cols) > max_cols_per_fig else ""
        prefix = FIG_DIR / f"umap_scores_{species}_{parent.replace(' ', '_')}{suffix}"
        plt.savefig(f"{prefix}.pdf", bbox_inches="tight")
        plt.savefig(f"{prefix}.png", bbox_inches="tight", dpi=200)
        plt.show()
    return cols


def run_validation(
    adata,
    marker_sets,
    parent,
    parent_values=None,
    parent_col="cell_type_level1_corrected",
    groupby="cell_type_level2",
    species="mouse",
    min_genes=1,
    max_cells_for_plot=None,
    plot_scores_umap=True,
):
    """筛选父类细胞、过滤 marker、计算 signature score，并输出验证图表。"""
    if max_cells_for_plot is None:
        max_cells_for_plot = globals().get("MAX_CELLS_FOR_PLOT", 120_000)
    parent_values = parent_values if parent_values is not None else [parent]
    if parent not in marker_sets:
        raise KeyError(f"{parent!r} 不在 marker_sets 中。可选父类: {list(marker_sets)}")
    adata_parent = subset_parent(adata, parent_values=parent_values, parent_col=parent_col, copy=True)
    print(f"筛选后数据维度: {adata_parent.shape}")
    print(f"父类细胞取值: {parent_values}")
    if adata_parent.n_obs == 0:
        raise ValueError(f"没有筛选到细胞。请检查 {parent_col=} 和 {parent_values=}")
    if groupby in adata_parent.obs.columns:
        print(adata_parent.obs[groupby].astype(str).value_counts().head(50).to_string())
    else:
        print(f"警告: {groupby!r} 不在 obs 中。可用列示例:", list(adata_parent.obs.columns)[:30])

    marker_dict, report = filter_marker_dict(marker_sets[parent], adata_parent.var_names, min_genes=min_genes)
    display(report)
    report.to_csv(TABLE_DIR / f"{species}_{parent.replace(' ', '_')}_marker_coverage.tsv", sep="\t", index=False)
    if not marker_dict:
        raise ValueError(f"{parent} 没有满足 min_genes={min_genes} 的 marker set，请降低 min_genes 或检查基因名。")

    adata_plot = downsample_for_plot(adata_parent, max_cells=max_cells_for_plot).copy()
    score_cols = add_signature_scores(adata_plot, marker_dict, score_prefix=f"{species}_{parent.replace(' ', '_')}")
    score_col_values = score_cols_list(score_cols)

    if groupby in adata_plot.obs.columns:
        plot_dotplot(
            adata_plot,
            marker_dict,
            groupby=groupby,
            title=f"{species} {parent}: 文献 marker-high 候选",
            save_path=FIG_DIR / f"dotplot_{species}_{parent.replace(' ', '_')}_{groupby}.pdf",
        )
        score_df = score_summary_by_group(adata_plot, score_col_values, groupby=groupby)
        display(score_df)
        score_df.to_csv(TABLE_DIR / f"{species}_{parent.replace(' ', '_')}_signature_scores_by_{groupby}.tsv", sep="\t")
    else:
        score_df = pd.DataFrame()

    if plot_scores_umap:
        plot_score_umap(adata_plot, score_cols, parent=parent, species=species)

    return adata_plot, marker_dict, score_cols, report, score_df

## 验证 1：T cell 重命名候选

字典中的候选包括 CD4 T cell、CD8 T cell、Treg、proliferating Treg/T cell、activated/memory T cell 和 senescent T cell。

In [ ]:
show_evidence(parent_keyword='T|CD4|CD8|Treg')

In [ ]:
adata_t_mouse, t_markers_mouse, t_score_cols_mouse, t_report_mouse, t_scores_mouse = run_validation(
    adata_mouse,
    marker_sets=MOUSE_MARKER_SETS,
    parent='T cell',
    parent_values=['T cell'],
    parent_col="cell_type_level1_corrected",
    groupby='cell_type_level2',
    species="mouse",
    min_genes=MIN_GENES,
)

In [ ]:
STATE = 'Foxp3-Il2ra-high Treg'
score_col = t_score_cols_mouse[STATE]
plot_umap_markers_and_scores(
    adata_t_mouse,
    t_markers_mouse,
    state=STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_T_cell_{safe_state_name(STATE)}",
)

## 验证 2：Macrophage 重命名候选

字典中的候选包括 resident/phagocytic macrophage、inflammatory macrophage、IFNIC、TREM2/CSF1 plaque macrophage 和 efferocytic macrophage 等。

In [ ]:
show_evidence(parent_keyword='macrophage|Mphi')

In [ ]:
adata_mac_mouse, mac_markers_mouse, mac_score_cols_mouse, mac_report_mouse, mac_scores_mouse = run_validation(
    adata_mouse,
    marker_sets=MOUSE_MARKER_SETS,
    parent='Macrophage',
    parent_values=['Macrophage'],
    parent_col="cell_type_level1_corrected",
    groupby='cell_type_level3',
    species="mouse",
    min_genes=MIN_GENES,
)

In [ ]:
STATE = 'Trem2-Csf1-Spp1-high plaque macrophage'
score_col = mac_score_cols_mouse[STATE]
plot_umap_markers_and_scores(
    adata_mac_mouse,
    mac_markers_mouse,
    state=STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_Macrophage_{safe_state_name(STATE)}",
)

## 验证 3：Monocyte 重命名候选

字典中的候选包括 classical monocyte、non-classical monocyte 和 inflammatory transitional monocyte。

In [ ]:
show_evidence(parent_keyword='monocyte|Ly6C|myeloid')

In [ ]:
adata_mono_mouse, mono_markers_mouse, mono_score_cols_mouse, mono_report_mouse, mono_scores_mouse = run_validation(
    adata_mouse,
    marker_sets=MOUSE_MARKER_SETS,
    parent='Monocyte',
    parent_values=['Monocyte'],
    parent_col="cell_type_level1_corrected",
    groupby='cell_type_level2',
    species="mouse",
    min_genes=MIN_GENES,
)

In [ ]:
STATE = 'Ly6c2-Ccr2-high classical monocyte'
score_col = mono_score_cols_mouse[STATE]
plot_umap_markers_and_scores(
    adata_mono_mouse,
    mono_markers_mouse,
    state=STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_Monocyte_{safe_state_name(STATE)}",
)

## 验证 4：SMC 重命名候选

字典中的候选包括 fibromyocyte/modulated SMC、SEM/intermediate SMC、contractile SMC、senescent VSMC、pyroptotic/senescent VSMC 和 fibrous-cap VSMC 等。

In [ ]:
show_evidence(parent_keyword='SMC|Smooth|VSMC|fibromyocyte')

In [ ]:
adata_smc_mouse, smc_markers_mouse, smc_score_cols_mouse, smc_report_mouse, smc_scores_mouse = run_validation(
    adata_mouse,
    marker_sets=MOUSE_MARKER_SETS,
    parent='Smooth muscle cell',
    parent_values=['Smooth muscle cell'],
    parent_col="cell_type_level1_corrected",
    groupby='cell_type_level2',
    species="mouse",
    min_genes=MIN_GENES,
)

In [ ]:
STATE = 'Lum-Fn1-high fibromyocyte/modulated SMC'
score_col = smc_score_cols_mouse[STATE]
plot_umap_markers_and_scores(
    adata_smc_mouse,
    smc_markers_mouse,
    state=STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_SMC_{safe_state_name(STATE)}",
)

## 验证 5：Endothelial cell 重命名候选

字典中的候选包括 inflammatory EC、EndMT EC、injury EC、disease-associated EC、EC1 和 AP-1 EC program。

In [ ]:
show_evidence(parent_keyword='Endothelial|EndMT|EC')

In [ ]:
adata_ec_mouse, ec_markers_mouse, ec_score_cols_mouse, ec_report_mouse, ec_scores_mouse = run_validation(
    adata_mouse,
    marker_sets=MOUSE_MARKER_SETS,
    parent='Endothelial cell',
    parent_values=['Endothelial cell'],
    parent_col="cell_type_level1_corrected",
    groupby='cell_type_level3',
    species="mouse",
    min_genes=MIN_GENES,
)

In [ ]:
STATE = 'Ccl2-Icam1-Vcam1-high inflammatory EC'
score_col = ec_score_cols_mouse[STATE]
plot_umap_markers_and_scores(
    adata_ec_mouse,
    ec_markers_mouse,
    state=STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_Endothelial_cell_{safe_state_name(STATE)}",
)

## 验证 6：Mast cell 重命名候选

字典中的候选为 Kit/Hdc/Cma1-high mast cell。

In [ ]:
show_evidence(parent_keyword='mast')

In [ ]:
adata_mast_mouse, mast_markers_mouse, mast_score_cols_mouse, mast_report_mouse, mast_scores_mouse = run_validation(
    adata_mouse,
    marker_sets=MOUSE_MARKER_SETS,
    parent='Mast cell',
    parent_values=['Mast cell'],
    parent_col="cell_type_level1_corrected",
    groupby='cell_type_level2',
    species="mouse",
    min_genes=MIN_GENES,
)

In [ ]:
STATE = 'Kit-Hdc-Cma1-high mast cell'
score_col = mast_score_cols_mouse[STATE]
plot_umap_markers_and_scores(
    adata_mast_mouse,
    mast_markers_mouse,
    state=STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_Mast_cell_{safe_state_name(STATE)}",
)

## 验证 7：Dendritic cell 重命名候选

字典中的候选包括 cDC1、cDC2 和 pDC。

In [ ]:
show_evidence(parent_keyword='dendritic|cDC|pDC')

In [ ]:
adata_dc_mouse, dc_markers_mouse, dc_score_cols_mouse, dc_report_mouse, dc_scores_mouse = run_validation(
    adata_mouse,
    marker_sets=MOUSE_MARKER_SETS,
    parent='Dendritic cell',
    parent_values=['Dendritic cell'],
    parent_col="cell_type_level1_corrected",
    groupby='cell_type_level3',
    species="mouse",
    min_genes=MIN_GENES,
)

In [ ]:
STATE = 'Xcr1-Clec9a-high cDC1'
score_col = dc_score_cols_mouse[STATE]
plot_umap_markers_and_scores(
    adata_dc_mouse,
    dc_markers_mouse,
    state=STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_Dendritic_cell_{safe_state_name(STATE)}",
)

## 验证 8：B cell 重命名候选

字典中的候选包括 Cd79a/Ms4a1-high B cell 和 Jchain/Mzb1-high plasma cell。

In [ ]:
show_evidence(parent_keyword='B cell|plasma|CD79')

In [ ]:
adata_b_mouse, b_markers_mouse, b_score_cols_mouse, b_report_mouse, b_scores_mouse = run_validation(
    adata_mouse,
    marker_sets=MOUSE_MARKER_SETS,
    parent='B cell',
    parent_values=['B cell'],
    parent_col="cell_type_level1_corrected",
    groupby='cell_type_level2',
    species="mouse",
    min_genes=MIN_GENES,
)

In [ ]:
STATE = 'Cd79a-Ms4a1-high B cell'
score_col = b_score_cols_mouse[STATE]
plot_umap_markers_and_scores(
    adata_b_mouse,
    b_markers_mouse,
    state=STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_B_cell_{safe_state_name(STATE)}",
)

## 验证 9：Natural killer cell 重命名候选

字典中的候选包括 NK cell 和 NK T-like cell。

In [ ]:
show_evidence(parent_keyword='NK|Natural killer|NKG7|ILC')

In [ ]:
adata_nk_mouse, nk_markers_mouse, nk_score_cols_mouse, nk_report_mouse, nk_scores_mouse = run_validation(
    adata_mouse,
    marker_sets=MOUSE_MARKER_SETS,
    parent='Natural killer cell',
    parent_values=['Natural killer cell'],
    parent_col="cell_type_level1_corrected",
    groupby='cell_type_level2',
    species="mouse",
    min_genes=MIN_GENES,
)

In [ ]:
STATE = 'Nkg7-Klrd1-high NK cell'
score_col = nk_score_cols_mouse[STATE]
plot_umap_markers_and_scores(
    adata_nk_mouse,
    nk_markers_mouse,
    state=STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_Natural_killer_cell_{safe_state_name(STATE)}",
)

## 验证 10：Fibroblast 重命名候选

字典中的候选包括 ECM fibroblast、proliferative/inflammatory fibroblast、senescent fibroblast、Smoc2-high disease-associated fibroblast 和 Fbn1-high fibroblast。

In [ ]:
show_evidence(parent_keyword='fibroblast|Fibro|FB')

In [ ]:
adata_fib_mouse, fib_markers_mouse, fib_score_cols_mouse, fib_report_mouse, fib_scores_mouse = run_validation(
    adata_mouse,
    marker_sets=MOUSE_MARKER_SETS,
    parent='Fibroblast',
    parent_values=['Fibroblast'],
    parent_col="cell_type_level1_corrected",
    groupby='cell_type_level2',
    species="mouse",
    min_genes=MIN_GENES,
)

In [ ]:
STATE = 'Col1a1-Col3a1-high ECM fibroblast'
score_col = fib_score_cols_mouse[STATE]
plot_umap_markers_and_scores(
    adata_fib_mouse,
    fib_markers_mouse,
    state=STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_Fibroblast_{safe_state_name(STATE)}",
)

## 自定义验证区块

当你想验证某一个父类细胞，或者某一个候选状态时，用这个区块。通常只需要改第一个 code cell 里的变量。

In [ ]:
# ---- 修改这里的变量 ----
TARGET_PARENT = "T cell"
TARGET_PARENT_VALUES = ["T cell"]
TARGET_PARENT_COL = "cell_type_level1_corrected"
TARGET_GROUPBY = "cell_type_level2"
TARGET_STATE = "Foxp3-Il2ra-high Treg"
# ------------------------

target_adata = adata_mouse
target_marker_sets = MOUSE_MARKER_SETS

adata_target, target_markers, target_score_cols, target_report, target_scores = run_validation(
    target_adata,
    marker_sets=target_marker_sets,
    parent=TARGET_PARENT,
    parent_values=TARGET_PARENT_VALUES,
    parent_col=TARGET_PARENT_COL,
    groupby=TARGET_GROUPBY,
    species="mouse_custom",
    min_genes=MIN_GENES,
)

print("可选候选状态:")
for k in target_markers:
    print("-", k)

In [ ]:
# 绘制一个指定候选状态的 marker 表达和 signature score UMAP。
if TARGET_STATE not in target_markers:
    raise KeyError(f"{TARGET_STATE!r} 不在 target_markers 中。可选状态: {list(target_markers)}")

score_col = target_score_cols[TARGET_STATE]
plot_umap_markers_and_scores(
    adata_target,
    target_markers,
    state=TARGET_STATE,
    score_col=score_col,
    max_marker_genes=9,
    save_prefix=FIG_DIR / f"umap_mouse_{safe_state_name(TARGET_PARENT)}_{safe_state_name(TARGET_STATE)}",
)

## 全图谱 UMAP：每个候选细胞类型的 marker 表达

上面的 UMAP 是在单个父类细胞 subset 中画 marker。这里新增全图谱验证：对 `MOUSE_MARKER_SETS` 中每个候选 marker-high 状态，在完整 `adata_mouse` 上计算 signature score，并在整个 atlas 的 UMAP 上查看：

- 候选状态的 signature score 是否集中在预期父类细胞区域。
- 关键 marker gene 在全图谱中是否只在目标区域表达，还是跨多个细胞类型表达。

默认会对所有候选状态生成全图谱 score UMAP；每个候选状态还会额外画最多 `MAX_GLOBAL_MARKER_GENES_PER_STATE` 个 marker gene + score。

In [ ]:
GLOBAL_FIG_DIR = FIG_DIR / "mouse_global_umap"
GLOBAL_TABLE_DIR = TABLE_DIR / "mouse_global_umap"
GLOBAL_FIG_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_TABLE_DIR.mkdir(parents=True, exist_ok=True)

# 全图谱细胞较多，默认下采样画图，避免一次画太慢。
MAX_GLOBAL_CELLS_FOR_PLOT = 180_000
MAX_GLOBAL_SCORE_PANELS_PER_FIG = 12
MAX_GLOBAL_MARKER_GENES_PER_STATE = 6

# 如果只想先测试部分父类，改这里，例如 ["T cell", "Macrophage"]。
GLOBAL_PARENTS_TO_PLOT = list(MOUSE_MARKER_SETS.keys())
GLOBAL_PARENTS_TO_PLOT

In [ ]:
def flatten_marker_sets(marker_sets, parents=None):
    """把 parent -> state -> genes 展开成全图谱绘图表。"""
    parents = list(marker_sets) if parents is None else list(parents)
    rows = []
    flat = {}
    for parent in parents:
        for state, genes in marker_sets[parent].items():
            key = f"{parent} | {state}"
            flat[key] = list(dict.fromkeys(genes))
            rows.append({
                "parent": parent,
                "state": state,
                "global_state_key": key,
                "n_marker": len(flat[key]),
                "markers": ",".join(flat[key]),
            })
    return flat, pd.DataFrame(rows)


def filter_global_marker_sets(flat_marker_sets, var_names, min_genes=MIN_GENES):
    """过滤全图谱中不存在的 marker。"""
    kept = {}
    rows = []
    var_names = pd.Index(var_names.astype(str) if hasattr(var_names, "astype") else var_names)
    for key, genes in flat_marker_sets.items():
        present = [g for g in genes if g in var_names]
        missing = [g for g in genes if g not in var_names]
        parent, state = key.split(" | ", 1)
        rows.append({
            "parent": parent,
            "state": state,
            "global_state_key": key,
            "n_total": len(genes),
            "n_present": len(present),
            "pct_present": len(present) / len(genes) if genes else 0,
            "present": ",".join(present),
            "missing": ",".join(missing),
        })
        if len(present) >= min_genes:
            kept[key] = present
    return kept, pd.DataFrame(rows)


def global_score_col_name(global_state_key):
    parent, state = global_state_key.split(" | ", 1)
    return f"global_score_{safe_state_name(parent)}_{safe_state_name(state)}"


def add_global_signature_scores(adata, global_marker_dict):
    """在全图谱 adata 上为每个候选状态计算 score。"""
    score_cols = {}
    for key, genes in global_marker_dict.items():
        col = global_score_col_name(key)
        sc.tl.score_genes(adata, gene_list=genes, score_name=col, use_raw=False)
        score_cols[key] = col
    return score_cols


def plot_global_score_umaps(adata, global_score_cols, group_name="all_candidates", max_panels=12):
    """在整个图谱 UMAP 上画所有候选状态 score。"""
    cols = list(global_score_cols.values())
    for i in range(0, len(cols), max_panels):
        chunk = cols[i:i + max_panels]
        sc.pl.umap(
            adata,
            color=chunk,
            cmap="viridis",
            ncols=3,
            frameon=False,
            wspace=0.3,
            save=False,
            show=False,
        )
        part = i // max_panels + 1
        prefix = GLOBAL_FIG_DIR / f"global_umap_scores_{safe_state_name(group_name)}_part{part}"
        plt.savefig(f"{prefix}.pdf", bbox_inches="tight")
        plt.savefig(f"{prefix}.png", bbox_inches="tight", dpi=200)
        plt.show()


def plot_global_marker_state_umaps(
    adata,
    global_marker_dict,
    global_score_cols,
    max_marker_genes=6,
):
    """在整个图谱 UMAP 上逐个候选状态画 marker genes + score。"""
    for key, genes in global_marker_dict.items():
        parent, state = key.split(" | ", 1)
        colors = genes[:max_marker_genes] + [global_score_cols[key]]
        print(f"全图谱 UMAP: {parent} | {state}")
        sc.pl.umap(
            adata,
            color=colors,
            cmap="viridis",
            ncols=3,
            frameon=False,
            wspace=0.3,
            save=False,
            show=False,
        )
        prefix = GLOBAL_FIG_DIR / f"global_umap_markers_{safe_state_name(parent)}_{safe_state_name(state)}"
        plt.savefig(f"{prefix}.pdf", bbox_inches="tight")
        plt.savefig(f"{prefix}.png", bbox_inches="tight", dpi=200)
        plt.show()

In [ ]:
# 准备全图谱绘图对象。
# 注意：这里不是 subset 某个父类，而是在完整 atlas 上看候选状态分布。
adata_mouse_global = downsample_for_plot(
    adata_mouse,
    max_cells=MAX_GLOBAL_CELLS_FOR_PLOT,
    random_state=RANDOM_STATE,
).copy()
adata_mouse_global

In [ ]:
global_marker_sets_raw, global_marker_table = flatten_marker_sets(
    MOUSE_MARKER_SETS,
    parents=GLOBAL_PARENTS_TO_PLOT,
)
global_marker_sets, global_marker_coverage = filter_global_marker_sets(
    global_marker_sets_raw,
    adata_mouse_global.var_names,
    min_genes=MIN_GENES,
)

global_marker_table.to_csv(GLOBAL_TABLE_DIR / "global_mouse_candidate_marker_sets.tsv", sep="\t", index=False)
global_marker_coverage.to_csv(GLOBAL_TABLE_DIR / "global_mouse_candidate_marker_coverage.tsv", sep="\t", index=False)

print("候选状态总数:", len(global_marker_sets_raw))
print("满足 min_genes 的候选状态数:", len(global_marker_sets))
display(global_marker_coverage)

In [ ]:
global_score_cols = add_global_signature_scores(adata_mouse_global, global_marker_sets)

global_score_meta = pd.DataFrame([
    {
        "parent": key.split(" | ", 1)[0],
        "state": key.split(" | ", 1)[1],
        "global_state_key": key,
        "score_col": col,
    }
    for key, col in global_score_cols.items()
])
global_score_meta.to_csv(GLOBAL_TABLE_DIR / "global_mouse_candidate_score_columns.tsv", sep="\t", index=False)
display(global_score_meta)

In [ ]:
# 全图谱上画所有候选状态的 signature score UMAP。
plot_global_score_umaps(
    adata_mouse_global,
    global_score_cols,
    group_name="all_marker_high_candidates",
    max_panels=MAX_GLOBAL_SCORE_PANELS_PER_FIG,
)

In [ ]:
# 全图谱上逐个候选状态画 marker genes + signature score。
# 图很多，但这是最直接判断每个候选状态是否只落在预期区域的方法。
plot_global_marker_state_umaps(
    adata_mouse_global,
    global_marker_sets,
    global_score_cols,
    max_marker_genes=MAX_GLOBAL_MARKER_GENES_PER_STATE,
)

## 全图谱 UMAP 自定义查看

如果只想看某一个候选状态在整个 atlas 上的 marker 表达和 score 分布，修改下面 `GLOBAL_TARGET_PARENT` 和 `GLOBAL_TARGET_STATE`。

In [ ]:
GLOBAL_TARGET_PARENT = "T cell"
GLOBAL_TARGET_STATE = "Foxp3-Il2ra-high Treg"
GLOBAL_TARGET_KEY = f"{GLOBAL_TARGET_PARENT} | {GLOBAL_TARGET_STATE}"

if GLOBAL_TARGET_KEY not in global_marker_sets:
    raise KeyError(f"{GLOBAL_TARGET_KEY!r} 不在 global_marker_sets 中。可选: {list(global_marker_sets)[:10]} ...")

plot_global_marker_state_umaps(
    adata_mouse_global,
    {GLOBAL_TARGET_KEY: global_marker_sets[GLOBAL_TARGET_KEY]},
    {GLOBAL_TARGET_KEY: global_score_cols[GLOBAL_TARGET_KEY]},
    max_marker_genes=MAX_GLOBAL_MARKER_GENES_PER_STATE,
)

## rename 判断记录

一个候选 rename 更可靠，通常需要同时满足：

- 大部分 marker 基因都存在于当前 AnnData 对象中。
- DotPlot 显示这一组 marker 在一个或少数几个现有 cluster 中富集。
- UMAP 上单基因表达和 signature score 的空间分布基本重叠。
- 该候选在证据表中属于 `全文支持` 或 `全文支持但需验证` 时优先级更高。

不要只根据弱证据行直接做最终 rename。`摘要/PubMed级` 和 `摘要/GEO级` 可以作为候选方向，但需要你自己的 DotPlot/UMAP 支持后再决定。